# Analyse Multidimensionnelle — ACP, ACM et Classification Non Supervisée
## DU Big Data & Data Science — Université de Montpellier 2025-2026
### Équipe : Randriamisaina Tsiory-Fanomezana · SHIRALI POUR Amir

**Plan :**
1. Préparation des données
2. ACP — Analyse en Composantes Principales (variables numériques)
3. ACM — Analyse des Correspondances Multiples (variables catégorielles)
4. Classification Hiérarchique Ascendante (CAH) — dendrogramme
5. Classification K-Means — profils des clusters
6. Synthèse

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def _racine():
    rep = Path().resolve()
    while True:
        if any((rep / m).exists() for m in ['.git', 'requirements.txt']):
            return rep
        if rep.parent == rep:
            return Path().resolve().parent
        rep = rep.parent

RACINE = _racine()
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
sns.set_style('whitegrid')
BLEU = '#4472C4'; ORANGE = '#ED7D31'; VERT = '#70AD47'; ROUGE = '#C00000'
print(f'Racine : {RACINE}')

## 1. Préparation des données

In [ ]:
df = pd.read_csv(RACINE / 'data' / 'dataset_complet.csv', encoding='utf-8')
df['date.ouverture'] = pd.to_datetime(df['date.ouverture'], errors='coerce')

# Variables numériques pour l'ACP
VARS_NUM = [
    'duree_totale_h',
    'agent_experience_j',
    'agent_duree_travail_j',
    'agent_temps_travail_pct',
    'nb_interventions',
    'nb_agents_distincts',
    'delai_survenance_ouverture_j',
    'mois_ouverture',
]
VARS_NUM = [v for v in VARS_NUM if v in df.columns]

# Variables catégorielles pour l'ACM
VARS_CAT = ['Cause.intervention', 'agent_contrat', 'agent_lieu_travail', 'agent_population']
VARS_CAT = [v for v in VARS_CAT if v in df.columns]

# Sous-ensemble propre (sans NaN sur les variables numériques)
df_acp = df[VARS_NUM].dropna()
print(f'Dataset pour ACP : {df_acp.shape[0]:,} lignes × {len(VARS_NUM)} variables')

## 2. ACP — Analyse en Composantes Principales

In [ ]:
# Centrage-réduction (obligatoire avant ACP)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_acp)

# ACP complète
pca = PCA()
pca.fit(X_scaled)

# Variance expliquée
var_exp = pca.explained_variance_ratio_ * 100
var_cum = np.cumsum(var_exp)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, len(var_exp)+1), var_exp, color=BLEU, edgecolor='white')
axes[0].plot(range(1, len(var_exp)+1), var_exp, 'o-', color=ROUGE)
axes[0].axhline(y=100/len(VARS_NUM), color='gray', linestyle='--', label=f'Seuil Kaiser ({100/len(VARS_NUM):.1f}%)')
axes[0].set_xlabel('Composante principale')
axes[0].set_ylabel('Variance expliquée (%)')
axes[0].set_title('Scree plot — Variance par composante', fontweight='bold')
axes[0].legend()

# Variance cumulée
axes[1].plot(range(1, len(var_cum)+1), var_cum, 'o-', color=BLEU, linewidth=2)
axes[1].axhline(y=80, color=ORANGE, linestyle='--', label='Seuil 80%')
axes[1].set_xlabel('Nombre de composantes')
axes[1].set_ylabel('Variance cumulée (%)')
axes[1].set_title('Variance cumulée expliquée', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Variance expliquée par composante :')
for i, (v, c) in enumerate(zip(var_exp, var_cum)):
    print(f'  CP{i+1}: {v:.1f}% (cumulé: {c:.1f}%)')

In [ ]:
# Cercle des corrélations (CP1 vs CP2)
n_components = 2
pca2 = PCA(n_components=n_components)
coords = pca2.fit_transform(X_scaled)

# Corrélations variables × composantes
correlations = pca2.components_.T * np.sqrt(pca2.explained_variance_)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Cercle des corrélations ---
ax = axes[0]
cercle = plt.Circle((0, 0), 1, fill=False, color='gray', linewidth=1)
ax.add_patch(cercle)
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)

for i, var in enumerate(VARS_NUM):
    ax.annotate('', xy=(correlations[i, 0], correlations[i, 1]),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=BLEU, lw=2))
    offset = 0.05
    ax.text(correlations[i, 0] + offset, correlations[i, 1] + offset,
            var.replace('agent_', '').replace('_j', '').replace('_h', '').replace('_pct', '%'),
            fontsize=9, color=ROUGE)

ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
ax.axvline(0, color='gray', linestyle='--', linewidth=0.5)
ax.set_xlabel(f'CP1 ({var_exp[0]:.1f}%)')
ax.set_ylabel(f'CP2 ({var_exp[1]:.1f}%)')
ax.set_title('Cercle des corrélations', fontweight='bold')
ax.set_aspect('equal')

# --- Projection des individus (échantillon) ---
ax2 = axes[1]
sample_idx = np.random.default_rng(42).choice(len(coords), size=min(2000, len(coords)), replace=False)
x_s, y_s = coords[sample_idx, 0], coords[sample_idx, 1]
ax2.scatter(x_s, y_s, alpha=0.15, s=5, color=BLEU)
ax2.axhline(0, color='gray', linestyle='--', linewidth=0.5)
ax2.axvline(0, color='gray', linestyle='--', linewidth=0.5)
ax2.set_xlabel(f'CP1 ({var_exp[0]:.1f}%)')
ax2.set_ylabel(f'CP2 ({var_exp[1]:.1f}%)')
ax2.set_title('Projection des individus (n=2000)', fontweight='bold')

plt.tight_layout()
plt.show()

# Contributions
print('\nContributions des variables aux composantes principales :')
contrib_df = pd.DataFrame(
    np.abs(correlations)**2 / np.sum(np.abs(correlations)**2, axis=0) * 100,
    index=VARS_NUM, columns=[f'CP{i+1}' for i in range(n_components)]
).round(1)
print(contrib_df.to_string())

## 3. ACM — Analyse des Correspondances Multiples

In [ ]:
# ACM via encodage one-hot + ACP sur tableau disjonctif complet
df_cat = df[VARS_CAT].dropna()
print(f'Dataset ACM : {df_cat.shape[0]:,} lignes')

# Tableau disjonctif complet
df_dummies = pd.get_dummies(df_cat, dtype=float)
# Centrage par la moyenne (MCA)
Z = df_dummies - df_dummies.mean()
Z = Z / np.sqrt(df_dummies.mean() * (1 - df_dummies.mean()))
Z = Z.fillna(0)

pca_mca = PCA(n_components=2)
coords_mca = pca_mca.fit_transform(Z)
var_exp_mca = pca_mca.explained_variance_ratio_ * 100

# Coordonnées des modalités
modalite_coords = pca_mca.components_.T

fig, ax = plt.subplots(figsize=(12, 8))

# Nuage des individus (échantillon)
sample_idx = np.random.default_rng(42).choice(len(coords_mca), size=min(1500, len(coords_mca)), replace=False)
ax.scatter(coords_mca[sample_idx, 0], coords_mca[sample_idx, 1],
           alpha=0.08, s=4, color='lightblue', label='Individus')

# Positions des modalités
couleurs_var = [BLEU, ORANGE, VERT, ROUGE]
col_names = df_dummies.columns.tolist()
for i, (var, coul) in enumerate(zip(VARS_CAT, couleurs_var)):
    cols_var = [c for c in col_names if c.startswith(var + '_') or c == var]
    for col in cols_var:
        idx = col_names.index(col)
        label = col.replace(var + '_', '')
        ax.scatter(modalite_coords[idx, 0], modalite_coords[idx, 1],
                   s=80, color=coul, zorder=5)
        ax.annotate(label, (modalite_coords[idx, 0], modalite_coords[idx, 1]),
                    fontsize=8, color=coul,
                    xytext=(5, 5), textcoords='offset points')

ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
ax.axvline(0, color='gray', linestyle='--', linewidth=0.5)
ax.set_xlabel(f'Axe 1 ({var_exp_mca[0]:.1f}%)')
ax.set_ylabel(f'Axe 2 ({var_exp_mca[1]:.1f}%)')
ax.set_title('ACM — Carte des modalités catégorielles', fontweight='bold')

legend_patches = [mpatches.Patch(color=c, label=v) for v, c in zip(VARS_CAT, couleurs_var)]
ax.legend(handles=legend_patches, loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()
print(f'Variance expliquée : Axe1={var_exp_mca[0]:.1f}%, Axe2={var_exp_mca[1]:.1f}%')